# ABSA Agent Pipeline — Google Colab

This notebook runs the **Aspect-Based Sentiment Analysis (ABSA)** multi-agent pipeline on Google Colab.

The pipeline processes restaurant customer reviews through **6 sequential stages**:
1. **Toxicity Detection** — Screens for toxic/abusive content
2. **Aspect Extraction** — Identifies restaurant experience aspects
3. **Sentiment Analysis** — Classifies sentiment per aspect
4. **Policy Evaluation & Escalation** — Checks thresholds and creates tickets
5. **Response Generation** — Composes customer-facing response
6. **Output Guardrail** — Validates response before publishing

---

## 1. Install Dependencies

In [ ]:
!pip install -q google-adk>=1.2.0 litellm>=1.40.0 python-dotenv>=1.0.0

## 2. Configure API Keys

Set your LLM provider API key below. You need **at least one**.

Update `MODEL_NAME` to match the provider you configured.

In [ ]:
import os
from google.colab import userdata

# --- Option A: Use Colab Secrets (recommended) ---
# Add your API key in Colab: click the key icon in the left sidebar
# then reference it here:
# os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
# os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
# os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')

# --- Option B: Paste key directly (less secure) ---
os.environ['OPENAI_API_KEY'] = 'your-api-key-here'  # <-- Replace this

# --- Choose your model ---
# Supported: openai/gpt-4o, anthropic/claude-3-5-sonnet-20241022,
#            google/gemini-2.0-flash, groq/llama3-8b-8192, xai/grok-2
MODEL_NAME = 'openai/gpt-4o'

print(f'Model: {MODEL_NAME}')

## 3. Create Project Structure

Creates all project files directly in Colab's filesystem.

In [ ]:
import os

PROJECT_ROOT = '/content/absa'
os.makedirs(f'{PROJECT_ROOT}/agents', exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/pipeline', exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/data', exist_ok=True)

import sys
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f'Project root: {PROJECT_ROOT}')

### 3.1 Write Sample Reviews Data

In [ ]:
import json

sample_reviews = [
  {
    "review_id": "REV-001",
    "restaurant_id": "REST-1042",
    "customer_id": "CUST-501",
    "customer_name": "Sarah Mitchell",
    "customer_tier": "Platinum",
    "review_text": "Absolutely loved the grilled salmon \u2014 perfectly cooked and beautifully plated. The waiter, James, was attentive and made great wine recommendations. The ambiance was cozy with perfect lighting for our anniversary dinner. Will definitely be back!",
    "review_timestamp": "2026-02-28T19:30:00Z",
    "source_platform": "google_reviews"
  },
  {
    "review_id": "REV-002",
    "restaurant_id": "REST-1042",
    "customer_id": "CUST-502",
    "customer_name": "David Chen",
    "customer_tier": "Gold",
    "review_text": "The pasta was bland and overcooked. I waited 45 minutes for my entree and the server never checked on us once. Very disappointing for the price point.",
    "review_timestamp": "2026-02-28T20:15:00Z",
    "source_platform": "yelp"
  },
  {
    "review_id": "REV-003",
    "restaurant_id": "REST-1055",
    "customer_id": "CUST-503",
    "customer_name": "Maria Rodriguez",
    "customer_tier": "Standard",
    "review_text": "Food was okay, nothing special. The chicken was a bit dry but the sides were tasty. Service was average \u2014 not bad but not memorable either.",
    "review_timestamp": "2026-02-27T12:45:00Z",
    "source_platform": "tripadvisor"
  },
  {
    "review_id": "REV-004",
    "restaurant_id": "REST-1042",
    "customer_id": "CUST-504",
    "customer_name": "James O'Brien",
    "customer_tier": "Silver",
    "review_text": "This place is absolute garbage. The food was disgusting and the staff are incompetent morons who shouldn't be allowed near a restaurant. Complete waste of money, I hope this dump gets shut down.",
    "review_timestamp": "2026-02-27T21:00:00Z",
    "source_platform": "google_reviews"
  },
  {
    "review_id": "REV-005",
    "restaurant_id": "REST-1055",
    "customer_id": "CUST-505",
    "customer_name": "Emily Watson",
    "customer_tier": "Platinum",
    "review_text": "The steak was phenomenal \u2014 best I've had in years. However, the restroom was filthy and there was a sticky residue on our table. For a restaurant at this price point, cleanliness should be a given.",
    "review_timestamp": "2026-02-26T19:00:00Z",
    "source_platform": "yelp"
  }
]

with open(f'{PROJECT_ROOT}/data/sample_reviews.json', 'w') as f:
    json.dump(sample_reviews, f, indent=2)

print(f'Written {len(sample_reviews)} sample reviews')

### 3.2 Write Brand Voice Guidelines

In [ ]:
brand_guidelines = """# Brand Voice Guidelines \u2014 The Golden Fork Restaurant Group

## Our Voice
We are warm, genuine, and professional. Every response should feel like it comes from a caring team member who truly values the guest experience.

## Tone Principles
- **Empathetic**: We listen first. When a guest has a concern, we acknowledge their feelings before offering solutions.
- **Grateful**: We sincerely appreciate every guest who takes the time to share feedback.
- **Accountable**: We own our mistakes. No deflecting, no excuses.
- **Action-Oriented**: We don't just apologize \u2014 we explain what we're doing to make it right.
- **Personal**: We use the guest's name when available and reference specific details from their visit.

## Response Structure

### For Negative Reviews
1. Thank the guest for their feedback
2. Acknowledge the specific issue with empathy
3. Apologize sincerely
4. Describe a concrete action or remediation
5. If a callback is scheduled, mention that a team member will reach out (with timeframe)
6. Invite them back

### For Positive Reviews
1. Thank the guest warmly
2. Reference specific highlights they mentioned
3. Share their compliment will be passed to the team
4. Invite them to return

### For Mixed Reviews
1. Thank the guest
2. Celebrate the positives
3. Address negatives with the same care as fully negative reviews
4. Close with optimism

## Language Do's and Don'ts

### Do Use
- \"We appreciate you sharing this with us\"
- \"We're sorry to hear about your experience with...\"
- \"Our team is looking into this\"
- \"We'd love the chance to make it right\"
- \"Thank you for being a valued guest\"

### Don't Use
- \"Per our policy...\"
- \"Unfortunately, we cannot...\"
- \"You should have...\"
- \"That's not how it usually is\"
- Defensive language or blame-shifting
- Internal jargon or system terminology

## Sign-Off
Always close with: \"Warm regards, The [Restaurant Location] Team\"
"""

with open(f'{PROJECT_ROOT}/data/brand_voice_guidelines.md', 'w') as f:
    f.write(brand_guidelines)

print('Brand voice guidelines written.')

### 3.3 Write Agent Modules

In [ ]:
# --- Toxicity Detection Agent (Stage 1) ---

toxicity_agent_code = '''"""Toxicity Detection Agent (Stage 1)"""

from google.adk.agents import LlmAgent

TOXICITY_INSTRUCTION = """You are the Toxicity Detection Agent in a restaurant review processing pipeline.

## Your Task
Analyze the customer review and determine whether it contains toxic, abusive, or harmful content.

## Input
Review in session state under key `review_input` with: review_id, restaurant_id, customer_id, review_text, review_timestamp, source_platform

## Classification Rules
- Toxicity threshold: 0.7 (scores at or above this are considered toxic)
- Distinguish between genuine frustration/strong language (e.g., "the food was damn cold") and actual abuse (hate speech, threats, harassment, severe profanity directed at people)
- Rating-only reviews with no text: pass through with toxicity_score 0.0
- Categories to check: profanity, hate_speech, threat, harassment, personal_attack

## Output Format
Respond with ONLY a valid JSON object:
{{
  "is_toxic": false,
  "toxicity_score": 0.15,
  "toxicity_categories": [],
  "pipeline_action": "CONTINUE",
  "reasoning": "Brief explanation of classification"
}}

If toxic (score >= 0.7): set "is_toxic": true, "pipeline_action": "TERMINATE", list categories.
If not toxic: set "is_toxic": false, "pipeline_action": "CONTINUE".

Review to analyze:
{review_input}
"""

def create_toxicity_agent(model):
    return LlmAgent(
        name="toxicity_detection_agent", model=model,
        instruction=TOXICITY_INSTRUCTION,
        description="Detects toxic, abusive, or harmful content in customer reviews",
        output_key="toxicity_result")
'''

with open(f'{PROJECT_ROOT}/agents/toxicity_agent.py', 'w') as f:
    f.write(toxicity_agent_code)

print('toxicity_agent.py written.')

In [ ]:
# --- Aspect Extraction Agent (Stage 2) ---

aspect_agent_code = '''"""Aspect Extraction Agent (Stage 2)"""

from google.adk.agents import LlmAgent

ASPECT_EXTRACTION_INSTRUCTION = """You are the Aspect Extraction Agent in a restaurant review processing pipeline.

## Your Task
Extract all distinct aspects of the restaurant experience mentioned in the review.
Map each mention to the canonical taxonomy below.

## Prerequisite
First check the toxicity result: {toxicity_result}
If pipeline_action is "TERMINATE", respond with:
{{"aspects": [], "pipeline_action": "TERMINATE", "reason": "Review flagged as toxic"}}

## Canonical Taxonomy
| Code | Aspect | Example Mentions |
|------|--------|------------------|
| FOOD_QUALITY | Food Quality | taste, freshness, portion, cooking, ingredients, flavor, temperature |
| SERVICE | Customer Service | staff, waiter, server, rude, helpful, attentive, slow, manager |
| AMBIANCE | Ambiance | decor, noise, lighting, vibe, atmosphere, music, seating, view |
| CLEANLINESS | Cleanliness | clean, dirty, filthy, restroom, hygiene, sticky, crumbs |

## Extraction Rules
- Extract both explicit aspects ("the food was great") and implicit ones ("it was filthy in there" -> CLEANLINESS)
- Handle multi-aspect sentences: produce separate entries per aspect
- Confidence threshold: >= 0.6. Discard below-threshold extractions
- If zero aspects are extracted, assign a default GENERAL aspect and set "needs_manual_review": true

## Output Format
Respond with ONLY a valid JSON object:
{{
  "aspects": [
    {{
      "aspect_code": "FOOD_QUALITY",
      "mention_text": "the grilled salmon",
      "confidence": 0.95,
      "is_implicit": false
    }}
  ],
  "unrecognized_mentions": [],
  "needs_manual_review": false,
  "pipeline_action": "CONTINUE"
}}

Review to analyze:
{review_input}
"""

def create_aspect_extraction_agent(model):
    return LlmAgent(
        name="aspect_extraction_agent", model=model,
        instruction=ASPECT_EXTRACTION_INSTRUCTION,
        description="Extracts restaurant experience aspects from reviews using canonical taxonomy",
        output_key="aspect_result")
'''

with open(f'{PROJECT_ROOT}/agents/aspect_extraction_agent.py', 'w') as f:
    f.write(aspect_agent_code)

print('aspect_extraction_agent.py written.')

In [ ]:
# --- Sentiment Analysis Agent (Stage 3) ---

sentiment_agent_code = '''"""Sentiment Analysis Agent (Stage 3)"""

from google.adk.agents import LlmAgent

SENTIMENT_ANALYSIS_INSTRUCTION = """You are the Sentiment Analysis Agent in a restaurant review processing pipeline.

## Your Task
Classify the sentiment toward each extracted aspect from the previous stage.
A single review can be positive on one aspect and negative on another.

## Prerequisite
Check the toxicity result: {toxicity_result}
If pipeline_action is "TERMINATE", respond with:
{{"aspect_sentiments": [], "pipeline_action": "TERMINATE"}}

## Input
- Original review: {review_input}
- Extracted aspects: {aspect_result}

## Sentiment Labels & Scores
| Label | Score Range |
|-------|------------|
| VERY_POSITIVE | 0.6 to 1.0 |
| POSITIVE | 0.2 to 0.59 |
| NEUTRAL | -0.19 to 0.19 |
| NEGATIVE | -0.59 to -0.2 |
| VERY_NEGATIVE | -1.0 to -0.6 |

## Analysis Rules
- Handle negation: "not good" -> NEGATIVE
- Handle sarcasm: "Oh sure, the food was AMAZING" (with negative context) -> NEGATIVE
- Handle conditional sentiment: "would have been great if it weren't cold" -> NEGATIVE
- Include the supporting text span for each classification
- Compute overall_sentiment as a weighted average (equal weights by default)

## Output Format
Respond with ONLY a valid JSON object:
{{
  "aspect_sentiments": [
    {{
      "aspect_code": "FOOD_QUALITY",
      "sentiment_label": "POSITIVE",
      "sentiment_score": 0.75,
      "confidence": 0.9,
      "supporting_text": "the grilled salmon"
    }}
  ],
  "overall_sentiment": "POSITIVE",
  "overall_sentiment_score": 0.65,
  "pipeline_action": "CONTINUE"
}}
"""

def create_sentiment_analysis_agent(model):
    return LlmAgent(
        name="sentiment_analysis_agent", model=model,
        instruction=SENTIMENT_ANALYSIS_INSTRUCTION,
        description="Classifies sentiment per extracted aspect with scores and labels",
        output_key="sentiment_result")
'''

with open(f'{PROJECT_ROOT}/agents/sentiment_analysis_agent.py', 'w') as f:
    f.write(sentiment_agent_code)

print('sentiment_analysis_agent.py written.')

In [ ]:
# --- Escalation Tools ---

escalation_tools_code = '''"""Escalation Ticket File Tools"""

import json
import os
from datetime import datetime

TICKETS_FILE = os.path.join(os.path.dirname(__file__), "..", "data", "escalation_tickets.json")


def create_escalation_ticket(
    review_id: str,
    restaurant_id: str,
    customer_id: str,
    customer_tier: str,
    priority: str,
    sla_hours: int,
    primary_negative_aspect: str,
    review_summary: str
) -> str:
    """Create an escalation ticket by appending to the tickets file.

    Args:
        review_id: The review that triggered escalation.
        restaurant_id: The restaurant identifier.
        customer_id: The customer identifier.
        customer_tier: Customer loyalty tier (Platinum/Gold/Silver/Standard).
        priority: Ticket priority (HIGH/MEDIUM/LOW).
        sla_hours: SLA in hours for resolution.
        primary_negative_aspect: The main negative aspect code.
        review_summary: Brief summary of the review for ticket context.

    Returns:
        JSON string with ticket details.
    """
    os.makedirs(os.path.dirname(TICKETS_FILE), exist_ok=True)

    ticket = {
        "review_id": review_id,
        "restaurant_id": restaurant_id,
        "customer_id": customer_id,
        "customer_tier": customer_tier,
        "priority": priority,
        "sla_hours": sla_hours,
        "primary_negative_aspect": primary_negative_aspect,
        "review_summary": review_summary,
        "created_at": datetime.utcnow().isoformat()
    }

    with open(TICKETS_FILE, "a") as f:
        f.write(json.dumps(ticket) + "\\n")

    return json.dumps({
        "action": "CREATED",
        "priority": priority,
        "sla_hours": sla_hours,
        "message": f"New {priority} priority ticket created with {sla_hours}h SLA"
    })
'''

with open(f'{PROJECT_ROOT}/agents/escalation_tools.py', 'w') as f:
    f.write(escalation_tools_code)

print('escalation_tools.py written.')

In [ ]:
# --- Policy Evaluation & Escalation Agent (Stage 4) ---

policy_agent_code = '''"""Policy Evaluation & Escalation Agent (Stage 4)"""

from google.adk.agents import LlmAgent
from agents.escalation_tools import create_escalation_ticket

POLICY_ESCALATION_INSTRUCTION = """You are the Policy Evaluation & Escalation Agent in a restaurant review processing pipeline.

## Your Task
Evaluate the sentiment results against the rules below.
Raise notifications when thresholds are breached and create escalation tickets
for eligible customers based on their loyalty tier.

## Prerequisite
Check the toxicity result: {toxicity_result}
If pipeline_action is "TERMINATE", respond with:
{{"notifications": [], "escalation_ticket": null, "pipeline_action": "TERMINATE"}}

## Input
- Review metadata: {review_input}
- Sentiment results: {sentiment_result}

## Notification Rules
If any aspect has a sentiment_score below -0.5, raise a notification for that aspect.

## Escalation Rules
Determine customer tier from review metadata (customer_tier field).
Apply the matching tier rule:
- Platinum: Any negative aspect -> HIGH priority, 4h SLA
- Gold: Overall sentiment NEGATIVE or worse -> MEDIUM priority, 24h SLA
- Silver: Overall sentiment VERY_NEGATIVE -> LOW priority, 48h SLA
- Standard: No ticket

If escalation is needed, call `create_escalation_ticket` with the required parameters.

## Output Format
Respond with ONLY a valid JSON object:
{{
  "notifications": [
    {{
      "aspect_code": "SERVICE",
      "sentiment_score": -0.7,
      "message": "Sentiment score below threshold for SERVICE"
    }}
  ],
  "escalation_ticket": {{
    "ticket_id": 1,
    "action": "CREATED",
    "priority": "HIGH",
    "sla_hours": 4
  }},
  "pipeline_action": "CONTINUE"
}}

If no escalation needed, set "escalation_ticket": null.
"""

def create_policy_escalation_agent(model):
    return LlmAgent(
        name="policy_escalation_agent", model=model,
        instruction=POLICY_ESCALATION_INSTRUCTION,
        description="Evaluates policies, fires alerts, and creates escalation tickets",
        tools=[create_escalation_ticket],
        output_key="escalation_result")
'''

with open(f'{PROJECT_ROOT}/agents/policy_escalation_agent.py', 'w') as f:
    f.write(policy_agent_code)

print('policy_escalation_agent.py written.')

In [ ]:
# --- Response Generation Agent (Stage 5) ---

response_agent_code = '''"""Response Generation Agent (Stage 5)"""

import os
from google.adk.agents import LlmAgent

def _load_brand_guidelines() -> str:
    guide_path = os.path.join(os.path.dirname(__file__), "..", "data", "brand_voice_guidelines.md")
    try:
        with open(guide_path, "r") as f:
            return f.read()
    except FileNotFoundError:
        return "Use a warm, professional, and empathetic tone."

RESPONSE_GENERATION_INSTRUCTION = """You are the Response Generation Agent in a restaurant review processing pipeline.

## Your Task
Compose a personalized, brand-consistent response to the customer review.
Your response will be validated by the Output Guardrail Agent before publishing.

## Prerequisite
Check the toxicity result: {toxicity_result}
If pipeline_action is "TERMINATE", respond with:
{{"response_text": "", "pipeline_action": "TERMINATE", "reason": "Toxic review \u2014 no response generated"}}

## Input
- Review: {review_input}
- Aspects: {aspect_result}
- Sentiments: {sentiment_result}
- Escalation: {escalation_result}

## Brand Voice Guidelines
""" + _load_brand_guidelines() + """

## Response Rules
1. Address every negative aspect specifically \u2014 never ignore or gloss over complaints
2. For negative aspects: acknowledge -> apologize -> provide concrete next step
3. Acknowledge positives with genuine gratitude
4. If an escalation ticket was created, inform the customer that someone will reach out
   (include approximate timeframe based on SLA, but don't mention the ticket system)
5. Personalize: use customer name if available, reference specific review details
6. Maximum 300 words
7. NEVER disclose: sentiment scores, toxicity classifications, customer tier,
   policy triggers, or any system-internal data
8. Sign off with: "Warm regards, The [Restaurant] Team"

## Output Format
Respond with ONLY a valid JSON object:
{{
  "response_text": "Dear Sarah, thank you for sharing your wonderful experience...",
  "word_count": 150,
  "aspects_addressed": ["FOOD_QUALITY", "SERVICE", "AMBIANCE"],
  "callback_mentioned": false,
  "pipeline_action": "CONTINUE"
}}
"""

def create_response_generation_agent(model):
    return LlmAgent(
        name="response_generation_agent", model=model,
        instruction=RESPONSE_GENERATION_INSTRUCTION,
        description="Generates personalized, brand-consistent responses to customer reviews",
        output_key="response_result")
'''

with open(f'{PROJECT_ROOT}/agents/response_generation_agent.py', 'w') as f:
    f.write(response_agent_code)

print('response_generation_agent.py written.')

In [ ]:
# --- Output Guardrail Agent (Stage 6) ---

guardrail_agent_code = '''"""Output Guardrail Agent (Stage 6)"""

from google.adk.agents import LlmAgent

OUTPUT_GUARDRAIL_INSTRUCTION = """You are the Output Guardrail Agent in a restaurant review processing pipeline.

## Your Task
Validate the generated response against the guardrail rules below.
You are the final quality gate before publishing.

## Prerequisite
Check the toxicity result: {toxicity_result}
If pipeline_action is "TERMINATE", respond with:
{{"guardrail_passed": false, "guardrail_action": "SKIP", "reason": "Toxic review \u2014 no response to validate"}}

## Input
- Original review: {review_input}
- Aspects & sentiments: {aspect_result}, {sentiment_result}
- Escalation info: {escalation_result}
- Generated response: {response_result}

## Guardrail Rules

Check the generated response against EVERY rule below:

1. **Tone Consistency** (CRITICAL): Response must be empathetic for complaints and warm for praise. No passive-aggressive, dismissive, or overly casual language.
2. **Content Safety** (CRITICAL): No profanity, inflammatory language, or statements that could create legal liability (e.g., admissions of negligence).
3. **Completeness** (CRITICAL): Every aspect with negative sentiment must be addressed in the response. No complaints should be skipped.
4. **Length & Platform Rules** (WARNING): Response must be within 300 words.

## Severity Levels
- CRITICAL: Blocks publishing. Response must be revised.
- WARNING: Flags for review but may pass.

## Output Format
Respond with ONLY a valid JSON object:
{{
  "guardrail_passed": true,
  "violations": [],
  "guardrail_action": "PUBLISH",
  "validation_summary": "All checks passed. Response is ready for publishing."
}}

If violations found:
{{
  "guardrail_passed": false,
  "violations": [
    {{
      "rule_category": "Completeness",
      "description": "SERVICE aspect with negative sentiment was not addressed",
      "severity": "CRITICAL"
    }}
  ],
  "guardrail_action": "REVISE",
  "validation_summary": "1 critical violation found. Response needs revision."
}}

guardrail_action values: PUBLISH, REVISE, or ESCALATE_TO_HUMAN
"""

def create_output_guardrail_agent(model):
    return LlmAgent(
        name="output_guardrail_agent", model=model,
        instruction=OUTPUT_GUARDRAIL_INSTRUCTION,
        description="Validates generated responses against guardrail rules before publishing",
        output_key="guardrail_result")
'''

with open(f'{PROJECT_ROOT}/agents/output_guardrail_agent.py', 'w') as f:
    f.write(guardrail_agent_code)

print('output_guardrail_agent.py written.')

### 3.4 Write Package Init and Pipeline Orchestrator

In [ ]:
# --- agents/__init__.py ---

agents_init = '''"""ABSA Agent Modules"""

from agents.toxicity_agent import create_toxicity_agent
from agents.aspect_extraction_agent import create_aspect_extraction_agent
from agents.sentiment_analysis_agent import create_sentiment_analysis_agent
from agents.policy_escalation_agent import create_policy_escalation_agent
from agents.response_generation_agent import create_response_generation_agent
from agents.output_guardrail_agent import create_output_guardrail_agent

__all__ = [
    "create_toxicity_agent",
    "create_aspect_extraction_agent",
    "create_sentiment_analysis_agent",
    "create_policy_escalation_agent",
    "create_response_generation_agent",
    "create_output_guardrail_agent",
]
'''

with open(f'{PROJECT_ROOT}/agents/__init__.py', 'w') as f:
    f.write(agents_init)

# --- pipeline/__init__.py ---
with open(f'{PROJECT_ROOT}/pipeline/__init__.py', 'w') as f:
    f.write('')

print('Package init files written.')

In [ ]:
# --- pipeline/callbacks.py ---

callbacks_code = '''"""Pipeline callbacks for conditional flow control."""

import json
import os
from datetime import datetime
from google.genai import types

HUMAN_REVIEW_FILE = os.path.join(
    os.path.dirname(__file__), "..", "data", "human_review_queue.json"
)


def check_toxicity_before_agent(callback_context):
    """Skip remaining agents if the review was flagged as toxic.

    Used as before_agent_callback on stages 2-6.
    Returns Content to skip the agent if toxic, None to proceed normally.
    """
    state = callback_context.state
    toxicity_raw = state.get("toxicity_result", "{}")

    try:
        toxicity = json.loads(toxicity_raw) if isinstance(toxicity_raw, str) else toxicity_raw
    except (json.JSONDecodeError, TypeError):
        return None

    if toxicity.get("is_toxic", False):
        return types.Content(
            role="model",
            parts=[types.Part(text=\'{"pipeline_action": "TERMINATE", "reason": "Skipped — toxic review"}\')],
        )

    return None


def write_human_review_after_guardrail(callback_context):
    """Write failed responses to human review queue file.

    Used as after_agent_callback on the guardrail agent (stage 6).
    If guardrail_action is REVISE or ESCALATE_TO_HUMAN, appends to file.
    """
    state = callback_context.state
    guardrail_raw = state.get("guardrail_result", "{}")

    try:
        guardrail = json.loads(guardrail_raw) if isinstance(guardrail_raw, str) else guardrail_raw
    except (json.JSONDecodeError, TypeError):
        return None

    action = guardrail.get("guardrail_action", "PUBLISH")
    if action in ("REVISE", "ESCALATE_TO_HUMAN"):
        entry = {
            "review_input": state.get("review_input", "{}"),
            "response_result": state.get("response_result", "{}"),
            "guardrail_result": guardrail_raw,
            "guardrail_action": action,
            "violations": guardrail.get("violations", []),
            "queued_at": datetime.utcnow().isoformat(),
        }

        os.makedirs(os.path.dirname(HUMAN_REVIEW_FILE), exist_ok=True)
        with open(HUMAN_REVIEW_FILE, "a") as f:
            f.write(json.dumps(entry) + "\\n")

        print(f"  ⚠ Response queued for human review ({action})")

    return None
'''

with open(f'{PROJECT_ROOT}/pipeline/callbacks.py', 'w') as f:
    f.write(callbacks_code)

print('callbacks.py written.')

# --- pipeline/absa_pipeline.py ---

pipeline_code = \'\'\'"""ABSA Pipeline Orchestrator"""

import json
from typing import Optional

from google.adk.agents import SequentialAgent
from google.adk.models.lite_llm import LiteLlm
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.genai import types

from agents import (
    create_toxicity_agent,
    create_aspect_extraction_agent,
    create_sentiment_analysis_agent,
    create_policy_escalation_agent,
    create_response_generation_agent,
    create_output_guardrail_agent,
)
from pipeline.callbacks import check_toxicity_before_agent, write_human_review_after_guardrail

STAGE_NAMES = [
    "Toxicity Detection",
    "Aspect Extraction",
    "Sentiment Analysis",
    "Policy Evaluation & Escalation",
    "Response Generation",
    "Output Guardrail",
]


def create_absa_pipeline(model_name: str = "openai/gpt-4o") -> SequentialAgent:
    model = LiteLlm(model=model_name)

    # Stage 1: Toxicity — runs first, no before_callback needed
    toxicity = create_toxicity_agent(model)

    # Stages 2-5: Skip if toxic
    aspect = create_aspect_extraction_agent(model)
    aspect.before_agent_callback = check_toxicity_before_agent

    sentiment = create_sentiment_analysis_agent(model)
    sentiment.before_agent_callback = check_toxicity_before_agent

    escalation = create_policy_escalation_agent(model)
    escalation.before_agent_callback = check_toxicity_before_agent

    response = create_response_generation_agent(model)
    response.before_agent_callback = check_toxicity_before_agent

    # Stage 6: Guardrail — skip if toxic + write to human review if failed
    guardrail = create_output_guardrail_agent(model)
    guardrail.before_agent_callback = check_toxicity_before_agent
    guardrail.after_agent_callback = write_human_review_after_guardrail

    return SequentialAgent(
        name="absa_pipeline",
        description="ABSA Review Processing Pipeline — 6-stage with conditional flow",
        sub_agents=[toxicity, aspect, sentiment, escalation, response, guardrail],
    )


async def run_pipeline(
    review: dict,
    model_name: str = "openai/gpt-4o",
    pipeline: Optional[SequentialAgent] = None,
    session_service: Optional[InMemorySessionService] = None,
    runner: Optional[Runner] = None,
) -> dict:
    if pipeline is None:
        pipeline = create_absa_pipeline(model_name)
    if session_service is None:
        session_service = InMemorySessionService()
    if runner is None:
        runner = Runner(
            agent=pipeline,
            app_name="absa_pipeline_app",
            session_service=session_service,
        )

    review_id = review.get("review_id", "unknown")
    session = await session_service.create_session(
        app_name="absa_pipeline_app",
        user_id="pipeline_user",
        state={"review_input": json.dumps(review)},
    )

    print(f"\\n{\'=\'*60}")
    print(f"  Processing Review: {review_id}")
    print(f"  Restaurant: {review.get(\'restaurant_id\', \'N/A\')}")
    print(f"  Customer: {review.get(\'customer_name\', review.get(\'customer_id\', \'N/A\'))}")
    print(f"{\'=\'*60}")

    user_message = types.Content(
        role="user",
        parts=[types.Part(text=f"Process this review:\\n{json.dumps(review, indent=2)}")],
    )

    results = {}
    current_stage = 0

    async for event in runner.run_async(
        user_id="pipeline_user",
        session_id=session.id,
        new_message=user_message,
    ):
        if hasattr(event, "author") and event.author:
            stage_idx = _get_stage_index(event.author)
            if stage_idx is not None and stage_idx != current_stage:
                current_stage = stage_idx
                print(f"\\n  [{current_stage + 1}/6] {STAGE_NAMES[current_stage]}...")

        if event.is_final_response() and event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    results[getattr(event, "author", "unknown")] = part.text

    final_session = await session_service.get_session(
        app_name="absa_pipeline_app",
        user_id="pipeline_user",
        session_id=session.id,
    )
    state = final_session.state if final_session else {}

    pipeline_output = {
        "review_id": review_id,
        "toxicity_result": _safe_parse(state.get("toxicity_result", "{}")),
        "aspect_result": _safe_parse(state.get("aspect_result", "{}")),
        "sentiment_result": _safe_parse(state.get("sentiment_result", "{}")),
        "escalation_result": _safe_parse(state.get("escalation_result", "{}")),
        "response_result": _safe_parse(state.get("response_result", "{}")),
        "guardrail_result": _safe_parse(state.get("guardrail_result", "{}")),
    }

    _print_summary(pipeline_output)
    return pipeline_output


def _get_stage_index(agent_name: str):
    mapping = {
        "toxicity_detection_agent": 0,
        "aspect_extraction_agent": 1,
        "sentiment_analysis_agent": 2,
        "policy_escalation_agent": 3,
        "response_generation_agent": 4,
        "output_guardrail_agent": 5,
    }
    return mapping.get(agent_name)


def _safe_parse(text: str):
    if not text:
        return {}
    try:
        cleaned = text.strip()
        if cleaned.startswith("```"):
            lines = cleaned.split("\\n")
            cleaned = "\\n".join(lines[1:-1])
        return json.loads(cleaned)
    except (json.JSONDecodeError, TypeError):
        return {"raw_output": text}


def _print_summary(output: dict):
    print(f"\\n{\'─\'*60}")
    print(f"  Pipeline Summary for {output[\'review_id\']}")
    print(f"{\'─\'*60}")

    tox = output.get("toxicity_result", {})
    is_toxic = tox.get("is_toxic", False)
    tox_score = tox.get("toxicity_score", "N/A")
    action = tox.get("pipeline_action", "UNKNOWN")
    status = "TOXIC — Pipeline terminated" if is_toxic else "Clean"
    print(f"  Toxicity:    {status} (score: {tox_score}, action: {action})")

    if is_toxic:
        print(f"{\'─\'*60}\\n")
        return

    aspects = output.get("aspect_result", {})
    aspect_list = aspects.get("aspects", [])
    codes = [a.get("aspect_code", "?") for a in aspect_list]
    print(f"  Aspects:     {\', \'.join(codes) if codes else \'None extracted\'}")

    sents = output.get("sentiment_result", {})
    overall = sents.get("overall_sentiment", "N/A")
    overall_score = sents.get("overall_sentiment_score", "N/A")
    print(f"  Overall:     {overall} (score: {overall_score})")

    esc = output.get("escalation_result", {})
    ticket = esc.get("escalation_ticket")
    if ticket:
        print(f"  Escalation:  Ticket — {ticket.get(\'priority\', \'?\')} priority")
    else:
        print(f"  Escalation:  No ticket created")

    guard = output.get("guardrail_result", {})
    passed = guard.get("guardrail_passed", "N/A")
    g_action = guard.get("guardrail_action", "N/A")
    print(f"  Guardrail:   {\'PASSED\' if passed else \'FAILED\'} → {g_action}")

    resp = output.get("response_result", {})
    response_text = resp.get("response_text", "")
    if response_text:
        preview = response_text[:120] + "..." if len(response_text) > 120 else response_text
        print(f"  Response:    {preview}")

    print(f"{\'─\'*60}\\n")
\'\'\'

with open(f'{PROJECT_ROOT}/pipeline/absa_pipeline.py', 'w') as f:
    f.write(pipeline_code)

print('absa_pipeline.py written.')
print('\nAll project files created!')

## 4. Create the Pipeline

Build the SequentialAgent pipeline with your chosen LLM provider.

In [ ]:
from pipeline.absa_pipeline import create_absa_pipeline, run_pipeline
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner

print(f'Using model: {MODEL_NAME}')

pipeline = create_absa_pipeline(MODEL_NAME)
session_service = InMemorySessionService()
runner = Runner(
    agent=pipeline,
    app_name='absa_pipeline_app',
    session_service=session_service,
)

print(f'Pipeline created with {len(pipeline.sub_agents)} sub-agents:')
for i, agent in enumerate(pipeline.sub_agents):
    print(f'  [{i+1}] {agent.name}')

## 5. Load Sample Reviews

In [ ]:
import json

with open(f'{PROJECT_ROOT}/data/sample_reviews.json', 'r') as f:
    all_reviews = json.load(f)

print(f'Loaded {len(all_reviews)} sample reviews')
print(f'\nFirst review preview:')
print(json.dumps(all_reviews[0], indent=2))

## 6. Run Pipeline — Positive Review

Process a positive multi-aspect review (REV-001, Platinum customer).

In [ ]:
review = all_reviews[0]  # REV-001: Positive multi-aspect review
print(f'Processing: {review["review_id"]}')
print(f'Tier: {review["customer_tier"]}')
print(f'Text: {review["review_text"][:100]}...')

result = await run_pipeline(
    review,
    model_name=MODEL_NAME,
    pipeline=pipeline,
    session_service=session_service,
    runner=runner,
)

### Inspect Stage Results

In [ ]:
print('=== Toxicity Result ===')
print(json.dumps(result.get('toxicity_result', {}), indent=2))

print('\n=== Aspect Extraction ===')
print(json.dumps(result.get('aspect_result', {}), indent=2))

print('\n=== Sentiment Analysis ===')
print(json.dumps(result.get('sentiment_result', {}), indent=2))

In [ ]:
print('=== Escalation Result ===')
print(json.dumps(result.get('escalation_result', {}), indent=2))

print('\n=== Generated Response ===')
resp = result.get('response_result', {})
print(resp.get('response_text', 'No response generated'))

print('\n=== Guardrail Result ===')
print(json.dumps(result.get('guardrail_result', {}), indent=2))

## 7. Run Pipeline — Toxic Review

Demonstrate that toxic reviews are terminated at Stage 1 with no response.

In [ ]:
toxic_review = all_reviews[3]  # REV-004: Toxic review
print(f'Processing toxic review: {toxic_review["review_id"]}')
print(f'Text: {toxic_review["review_text"][:100]}...')

toxic_result = await run_pipeline(
    toxic_review,
    model_name=MODEL_NAME,
    pipeline=pipeline,
    session_service=session_service,
    runner=runner,
)

## 8. Run Pipeline — Negative Review with Escalation

Process a negative review from a Gold customer to trigger escalation.

In [ ]:
esc_review = all_reviews[1]  # REV-002: Negative review, Gold customer
print(f'Processing: {esc_review["review_id"]}')
print(f'Tier: {esc_review["customer_tier"]}')
print(f'Text: {esc_review["review_text"][:100]}...')

esc_result = await run_pipeline(
    esc_review,
    model_name=MODEL_NAME,
    pipeline=pipeline,
    session_service=session_service,
    runner=runner,
)

## 9. Run Pipeline — Mixed Review (Platinum)

Process a mixed review from a Platinum customer — positive food, negative cleanliness.

In [ ]:
mixed_review = all_reviews[4]  # REV-005: Mixed review, Platinum customer
print(f'Processing: {mixed_review["review_id"]}')
print(f'Tier: {mixed_review["customer_tier"]}')
print(f'Text: {mixed_review["review_text"][:100]}...')

mixed_result = await run_pipeline(
    mixed_review,
    model_name=MODEL_NAME,
    pipeline=pipeline,
    session_service=session_service,
    runner=runner,
)

## 10. Inspect Escalation Tickets

Check escalation tickets written to `data/escalation_tickets.json`.

In [ ]:
import os

tickets_path = f'{PROJECT_ROOT}/data/escalation_tickets.json'
if os.path.exists(tickets_path):
    with open(tickets_path, 'r') as f:
        lines = f.readlines()
    tickets = [json.loads(line) for line in lines if line.strip()]
    print(f'=== Escalation Tickets ({len(tickets)} total) ===')
    for ticket in tickets:
        print(json.dumps(ticket, indent=2))
        print()
else:
    print('No escalation tickets created yet.')

## 11. Inspect Human Review Queue

Check responses that were flagged by the guardrail and queued for human review.

In [ ]:
import json
import os

human_review_path = f'{PROJECT_ROOT}/data/human_review_queue.json'
if os.path.exists(human_review_path):
    with open(human_review_path, 'r') as f:
        lines = f.readlines()
    entries = [json.loads(line) for line in lines if line.strip()]
    print(f'=== Human Review Queue ({len(entries)} entries) ===')
    for entry in entries:
        print(json.dumps(entry, indent=2))
        print()
else:
    print('No responses queued for human review.')